# Toy Data — Step 2: Scene 1 / Scene 2 (Multi-Resolution Example)

This notebook generates the two toy rasters that reproduce the **Scene 1 / Scene 2** multi-resolution example described in:

> Pontius Jr., R.G. and Connors, J., 2009. **Range of categorical associations for comparison of maps with mixed pixels.** *Photogrammetric Engineering & Remote Sensing*, 75(8), pp.963-969. <https://www.ingentaconnect.com/content/asprs/pers/2009/00000075/00000008/art00003>

The example corresponds to Figure 1 of the paper, which is used to illustrate how the range of categorical association changes across resolutions (Figures 6 and 7).

Each scene is stored as a **single-band categorical raster**, `dtype=uint8`, with an integer class code per pixel (0=white, 1=black) — the same structure a real, hard-classified land-cover map would have.

Fractional membership is **not pre-computed or stored**. It only exists at the moment neighboring pixels are aggregated into a coarser grain (g=2, g=4, ...): within each block, the proportion of pixels belonging to each class is computed on demand. At the native resolution (g=1), there is no fraction at all — every pixel is 100% a single class, exactly as it should be.

In [ ]:
# -----------------------------------------------------------------------------
# 01. INSTALL REQUIRED LIBRARIES
# -----------------------------------------------------------------------------
!pip install rasterio -q

# -----------------------------------------------------------------------------
# 02. IMPORT LIBRARIES AND MOUNT GOOGLE DRIVE
# -----------------------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


In [ ]:
# -----------------------------------------------------------------------------
# 03. DEFINE SINGLE-BAND CATEGORICAL RASTER FUNCTION
# -----------------------------------------------------------------------------
def create_categorical_raster(
    class_grid: np.ndarray,
    output_path: str,
    pixel_size: float = 10.0,
    crs: str = "EPSG:4326"
) -> str:
    """
    Writes a single-band GeoTIFF where each pixel stores an integer class
    code (a pure, hard-classified category), matching the structure of a
    real categorical land-cover map: one raster per time point, one band,
    no fractional membership stored.

    Parameters
    ----------
    class_grid : np.ndarray
        2D array of integer class codes (rows x cols).
    output_path : str
        Full file path (including filename) for the output GeoTIFF.
    pixel_size : float, optional
        Spatial resolution of each pixel (default 10.0).
    crs : str, optional
        Coordinate Reference System string (default 'EPSG:4326').

    Returns
    -------
    str
        The output file path.
    """
    # 03.1 Ensure output directory exists
    os.makedirs(
        os.path.dirname(output_path),
        exist_ok=True
    )

    height, width = class_grid.shape

    # 03.2 Define spatial transform (origin arbitrary, at top-left)
    transform = from_origin(
        0.0,
        height * pixel_size,
        pixel_size,
        pixel_size
    )

    # 03.3 Define GeoTIFF metadata profile (single band, integer classes)
    meta = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": 1,
        "dtype": "uint8",
        "crs": crs,
        "transform": transform,
        "nodata": 255
    }

    # 03.4 Write raster file to disk
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(class_grid.astype(np.uint8), 1)

    print(f"[LOG] Exported: {output_path} (shape: {height}x{width}, 1 band, classes: {sorted(np.unique(class_grid).tolist())})")
    return output_path


In [ ]:
# -----------------------------------------------------------------------------
# 04. DATA INITIALIZATION (Figure 1 example of Pontius & Connors, 2009)
# -----------------------------------------------------------------------------

# 04.1 Class code convention for this toy example: 0 = white, 1 = black
CLASS_CODES = {"white": 0, "black": 1}
CLASSES = list(CLASS_CODES.values())

# 04.2 Scene 1: left half black, right half white (2x2 blocks stay pure)
scene1_grid = np.array([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
], dtype=np.uint8)

# 04.3 Scene 2: checkerboard (each 2x2 block is already 50/50 black/white)
scene2_grid = np.array([
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
], dtype=np.uint8)

# 04.4 Set output folder destination inside Google Drive
drive_output_folder = "/content/drive/MyDrive/toyData/step2_scenes"

# 04.5 Generate the two 4x4 single-band scene rasters
path_scene1 = create_categorical_raster(
    class_grid=scene1_grid,
    output_path=os.path.join(drive_output_folder, "scene1.tif"),
    pixel_size=10.0,
    crs="EPSG:4326"
)

path_scene2 = create_categorical_raster(
    class_grid=scene2_grid,
    output_path=os.path.join(drive_output_folder, "scene2.tif"),
    pixel_size=10.0,
    crs="EPSG:4326"
)

print("\n[SUCCESS] Both single-band categorical scene rasters have been generated and saved to Google Drive!")


## Verification — On-Demand Aggregation

Fractional membership is not read from a pre-computed multiband raster; it is calculated on demand, directly from the single-band categorical raster, for a chosen aggregation factor `g`.

`aggregate_categorical_to_membership()` does exactly this: for each class, it builds a binary mask (is this pixel that class? 1 or 0) and averages it within each `g x g` block. The result is the fraction of that class at each aggregated pixel, without ever needing to store a multiband raster on disk.

In [ ]:
# -----------------------------------------------------------------------------
# 05. ON-DEMAND AGGREGATION AND MULTI-RESOLUTION VERIFICATION
# -----------------------------------------------------------------------------
def read_categorical_raster(path: str) -> np.ndarray:
    """Reads a single-band categorical raster into a 2D array."""
    with rasterio.open(path) as src:
        return src.read(1)

def aggregate_categorical_to_membership(
    categorical_array: np.ndarray,
    classes: list,
    factor: int
) -> np.ndarray:
    """
    Aggregates a single-band categorical raster into fractional membership
    at a coarser grain, computed on demand (never pre-stored).

    Parameters
    ----------
    categorical_array : np.ndarray
        2D array of integer class codes.
    classes : list
        Ordered list of class codes to compute membership for.
    factor : int
        Aggregation factor (grain size g). Must evenly divide both
        dimensions of categorical_array.

    Returns
    -------
    np.ndarray
        Array of shape (len(classes), height/factor, width/factor) with
        the fractional membership of each class per aggregated pixel.
    """
    h, w = categorical_array.shape
    if h % factor != 0 or w % factor != 0:
        raise ValueError(
            f"factor={factor} does not evenly divide raster shape {(h, w)}"
        )
    h2, w2 = h // factor, w // factor
    membership = np.zeros((len(classes), h2, w2), dtype=np.float32)
    for idx, class_code in enumerate(classes):
        mask = (categorical_array == class_code).astype(np.float32)
        membership[idx] = mask.reshape(h2, factor, w2, factor).mean(axis=(1, 3))
    return membership

def map_level_stats(X: np.ndarray, Y: np.ndarray) -> tuple:
    """
    Computes map-level greatest (Eq. 2), random (Eq. 4), and least (Eq. 6)
    association for two membership arrays, assuming equal pixel weights Wn.
    """
    M = np.minimum(X, Y)
    R = X * Y
    L = np.maximum(0.0, X + Y - 1.0)
    return M.mean(), R.mean(), L.mean()

# 05.1 Read the categorical rasters back (round-trip check)
scene1_r = read_categorical_raster(path_scene1)
scene2_r = read_categorical_raster(path_scene2)

results = []
for factor in [1, 2, 4]:
    # 05.2 Aggregate on demand, for this factor only
    scene1_membership = aggregate_categorical_to_membership(scene1_r, CLASSES, factor)
    scene2_membership = aggregate_categorical_to_membership(scene2_r, CLASSES, factor)

    black_idx = CLASS_CODES["black"]
    white_idx = CLASS_CODES["white"]

    # Case A: Scene 1 black x Scene 2 white (Figure 6)
    X_a = scene1_membership[black_idx]
    Y_a = scene2_membership[white_idx]
    m_a, r_a, l_a = map_level_stats(X_a, Y_a)
    results.append({"case": "A: Scene1(black) x Scene2(white)", "g": factor, "M": m_a, "R": r_a, "L": l_a, "range": m_a - l_a})

    # Case B: Scene 2 black x Scene 2 white (Figure 7, map vs itself)
    X_b = scene2_membership[black_idx]
    Y_b = scene2_membership[white_idx]
    m_b, r_b, l_b = map_level_stats(X_b, Y_b)
    results.append({"case": "B: Scene2(black) x Scene2(white)", "g": factor, "M": m_b, "R": r_b, "L": l_b, "range": m_b - l_b})

results_df = pd.DataFrame(results)
display(results_df)

# 05.3 Qualitative pattern checks against the paper's description
case_a = results_df[results_df["case"].str.startswith("A")].set_index("g")
case_b = results_df[results_df["case"].str.startswith("B")].set_index("g")

print("\n[CHECK] Case A constant between g=1 and g=2:", np.isclose(case_a.loc[1, "M"], case_a.loc[2, "M"]))
print("[CHECK] Case A range opens up only at g=4 (range==0 at g=1,2; >0 at g=4):",
      np.isclose(case_a.loc[1, "range"], 0) and np.isclose(case_a.loc[2, "range"], 0) and case_a.loc[4, "range"] > 0)
print("[CHECK] Case B range is zero at g=1 (pure pixels, no overlap):", np.isclose(case_b.loc[1, "range"], 0))
print("[CHECK] Case B range jumps to >0 at g=2:", case_b.loc[2, "range"] > 0)
print("[CHECK] Case B range constant between g=2 and g=4:", np.isclose(case_b.loc[2, "range"], case_b.loc[4, "range"]))

# 05.4 Reference values, derived independently from the pixel-aggregation
# logic described in Figure 1 of Pontius & Connors (2009), for comparison
# against the values computed above
print("\n[REFERENCE] Expected values (derived independently from the aggregation logic):")
print("Case A -> g=1: M=0.250 R=0.250 L=0.250 | g=2: M=0.250 R=0.250 L=0.250 | g=4: M=0.500 R=0.250 L=0.000")
print("Case B -> g=1: M=0.000 R=0.000 L=0.000 | g=2: M=0.500 R=0.250 L=0.000 | g=4: M=0.500 R=0.250 L=0.000")
